In [ ]:
# Cell 1: load historical log output.
from pathlib import Path
from collections import deque
import json

REPO = Path('/workspace/stable-query-latent')
LOG = Path('/workspace/stable_query_latent_logs/pipeline.log')
SWEEP_MANIFEST = REPO / 'VICReg_review/heads/cloud_full_sweep_a100/sweep_manifest.json'
EMBED_MANIFEST = REPO / 'game_review_data/embedding_h5.h5.incloud_manifest.json'
TEXT_MANIFEST = REPO / 'game_review_data/build_new_gamedata/text_h5.h5.manifest.json'


def tail(path=LOG, lines=200):
    path = Path(path)
    print(f'log: {path}')
    if not path.exists():
        print('missing log file')
        return
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in deque(f, maxlen=lines):
            print(line, end='')


def show_manifest(path):
    path = Path(path)
    print(f'\n=== {path} ===')
    if not path.exists():
        print('missing')
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('bad json:', exc)
        return
    for key in ['status', 'updated_at', 'finished_at', 'epoch', 'step', 'error']:
        if key in data:
            print(f'{key}: {data[key]}')
    current = data.get('current')
    if current:
        print('current:', json.dumps(current, ensure_ascii=False))
    rows = data.get('rows')
    if isinstance(rows, list):
        done = sum(1 for row in rows if row.get('status') == 'done')
        print(f'combinations_done: {done}/{len(rows)}')


tail(lines=200)
show_manifest(TEXT_MANIFEST)
show_manifest(EMBED_MANIFEST)
show_manifest(SWEEP_MANIFEST)
HISTORY_END = LOG.stat().st_size if LOG.exists() else 0


In [ ]:
# Cell 2: realtime latest log output.
# Stop/interrupt this cell to stop watching. It does not stop the training job.
import time
from collections import deque
from pathlib import Path
from IPython.display import clear_output


START_OFFSET = globals().get('HISTORY_END', None)


def read_new_text(path, start_offset):
    if not path.exists():
        return start_offset, ''
    size = path.stat().st_size
    if start_offset is None or start_offset > size:
        start_offset = size
    with path.open('rb') as f:
        f.seek(start_offset)
        data = f.read()
        end_offset = f.tell()
    text = data.decode('utf-8', errors='replace')
    return end_offset, text


def follow(path=LOG, lines=120, interval=5, start_offset=START_OFFSET):
    path = Path(path)
    last_offset = start_offset
    seen_lines = deque(maxlen=lines)
    last_update = None
    while True:
        if path.exists():
            last_offset, new_text = read_new_text(path, last_offset)
            if new_text:
                seen_lines.extend(new_text.splitlines())
                last_update = time.strftime('%Y-%m-%d %H:%M:%S')
        else:
            new_text = ''

        clear_output(wait=True)
        print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} | {path}')
        print('-' * 100)
        if not path.exists():
            print('missing log file')
        elif seen_lines:
            if new_text:
                print(f'new output received at {last_update}')
            else:
                print(f'waiting for new output; last update at {last_update}')
            print('-' * 100)
            for line in seen_lines:
                print(line)
        else:
            print('waiting for new log lines after historical output')
        time.sleep(interval)


follow(lines=120, interval=5)


In [ ]:
# Cell 3: realtime system status monitor.
# Stop/interrupt this cell to stop watching. It does not stop the training job.
import subprocess, time
from pathlib import Path
from IPython.display import clear_output

try:
    import psutil
except ImportError:
    psutil = None
    print('psutil is missing. Run: pip install psutil')


def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == 'max':
            return None
        return int(text)
    except Exception:
        return None


def get_memory_status():
    limit = _read_int('/sys/fs/cgroup/memory.max')
    used = _read_int('/sys/fs/cgroup/memory.current')
    if limit is None or used is None:
        limit = _read_int('/sys/fs/cgroup/memory/memory.limit_in_bytes')
        used = _read_int('/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, 'cgroup'
    if psutil is None:
        return None, None, None, 'unavailable'
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, 'host'


def get_gpu_status():
    try:
        out = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw',
                '--format=csv,noheader,nounits',
            ],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        if not out:
            return ['n/a']
        rows = []
        for line in out.splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) >= 7:
                rows.append(
                    f'gpu{parts[0]} {parts[1]} | util {float(parts[2]):.0f}% | '
                    f'mem {float(parts[3]) / 1024:.1f}/{float(parts[4]) / 1024:.1f} GiB | '
                    f'temp {float(parts[5]):.0f}C | power {float(parts[6]):.0f}W'
                )
            else:
                rows.append(line)
        return rows
    except Exception as exc:
        return [f'n/a ({exc})']


def system_monitor(interval=5):
    if psutil is None:
        return
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while True:
        gpu_rows = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t

        clear_output(wait=True)
        print(time.strftime('%Y-%m-%d %H:%M:%S'))
        print('-' * 100)
        for row in gpu_rows:
            print(f'[gpu] {row}')
        if ram_pct is None:
            ram_msg = 'unavailable'
        else:
            ram_msg = f'{ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB, {ram_source})'
        print(f'[cpu] {cpu:.0f}%')
        print(f'[ram] {ram_msg}')
        print(f'[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s')
        time.sleep(interval)


system_monitor(interval=5)
